# TaylorMade product-gallery image puller

This workflow captures every full product-gallery image—such as the four SIM2 Max views—rather than only image resources that happen to load first. It supports either a TaylorMade category/listing link (process every product card on that page) or one product link.

1. Paste the listing or product link into `PRODUCT_URL` and run the first cell.
2. In the opened browser tab, run the DevTools script from the next cell. It opens one reusable product tab, visits each product item, scrolls its gallery to trigger lazy images, walks its visible product options, and copies every gallery URL.
3. Paste the copied list into `IMAGE_URLS_TEXT` in the final cell to download the files.

The script only changes visible product-option controls; it does not add anything to the cart. Keep the normal browser window open while it runs, and allow its one popup/tab if the browser asks. Use the downloaded images only as permitted by TaylorMade's terms and the applicable image rights.

In [12]:
from pathlib import Path
from urllib.parse import urlparse
import re

import requests


PROJECT_ROOT = next(
    parent
    for parent in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (parent / "requirements.txt").is_file()
)
# Change this classification when importing another kind of club.
OUTPUT_SUBDIRECTORY = Path("iron") / "wedges"
OUTPUT_DIR = PROJECT_ROOT / "assets" / "club-reference-images" / OUTPUT_SUBDIRECTORY
IMAGE_URL_PATTERN = re.compile(r"\.(?:jpg|jpeg|png|webp|avif)(?:\?.*)?$", re.IGNORECASE)


# Paste the DevTools collector's copied URLs here: one URL per line.
IMAGE_URLS_TEXT = """
https://www.taylormadegolf.com/dw/image/v2/AAIS_PRD/on/demandware.static/-/Sites-tmag-master-catalog/en_US/v1784520671511/zoom/N29140_zoom_D.jpg?sw=900&sh=900&sm=fit
https://www.taylormadegolf.com/dw/image/v2/AAIS_PRD/on/demandware.static/-/Sites-tmag-master-catalog/en_US/v1784520671511/zoom/N29140_zoom_D2.jpg?sw=900&sh=900&sm=fit
https://www.taylormadegolf.com/dw/image/v2/AAIS_PRD/on/demandware.static/-/Sites-tmag-master-catalog/en_US/v1784520671511/zoom/N29140_zoom_D3.jpg?sw=900&sh=900&sm=fit
https://www.taylormadegolf.com/dw/image/v2/AAIS_PRD/on/demandware.static/-/Sites-tmag-master-catalog/en_US/v1784520671511/zoom/N29140_zoom_D4.jpg?sw=900&sh=900&sm=fit
https://www.taylormadegolf.com/dw/image/v2/AAIS_PRD/on/demandware.static/-/Sites-tmag-master-catalog/en_US/v1784520671511/zoom/U20686_zoom_D.jpg?sw=900&sh=900&sm=fit
https://www.taylormadegolf.com/dw/image/v2/AAIS_PRD/on/demandware.static/-/Sites-tmag-master-catalog/en_US/v1784520671511/zoom/U20686_zoom_D.jpg?sw=900&sh=900&sm=fit
https://www.taylormadegolf.com/dw/image/v2/AAIS_PRD/on/demandware.static/-/Sites-tmag-master-catalog/en_US/v1784520671511/zoom/U20686_zoom_D2.jpg?sw=900&sh=900&sm=fit
https://www.taylormadegolf.com/dw/image/v2/AAIS_PRD/on/demandware.static/-/Sites-tmag-master-catalog/en_US/v1784520671511/zoom/U20686_zoom_D3.jpg?sw=900&sh=900&sm=fit
https://www.taylormadegolf.com/dw/image/v2/AAIS_PRD/on/demandware.static/-/Sites-tmag-master-catalog/en_US/v1784520671511/zoom/U20686_zoom_D4.jpg?sw=900&sh=900&sm=fit
https://www.taylormadegolf.com/dw/image/v2/AAIS_PRD/on/demandware.static/-/Sites-tmag-master-catalog/en_US/v1784520671511/zoom/N29140_zoom_D.jpg?sw=900&sh=900&sm=fit
https://www.taylormadegolf.com/dw/image/v2/AAIS_PRD/on/demandware.static/-/Sites-tmag-master-catalog/en_US/v1784520671511/zoom/TC693_zoom_D.jpg?sw=900&sh=900&sm=fit
https://www.taylormadegolf.com/dw/image/v2/AAIS_PRD/on/demandware.static/-/Sites-tmag-master-catalog/en_US/v1784520671511/zoom/TC693_zoom_D2.jpg?sw=900&sh=900&sm=fit
https://www.taylormadegolf.com/dw/image/v2/AAIS_PRD/on/demandware.static/-/Sites-tmag-master-catalog/en_US/v1784520671511/zoom/TC693_zoom_D3.jpg?sw=900&sh=900&sm=fit
https://www.taylormadegolf.com/dw/image/v2/AAIS_PRD/on/demandware.static/-/Sites-tmag-master-catalog/en_US/v1784520671511/zoom/TC693_zoom_D4.jpg?sw=900&sh=900&sm=fit
https://www.taylormadegolf.com/on/demandware.static/-/Sites-tmag-master-catalog/en_US/v1784520671511/zoom/TC441_zoom_D.jpg
https://www.taylormadegolf.com/on/demandware.static/-/Sites-tmag-master-catalog/en_US/v1784520671511/zoom/TC448_zoom_D.jpg
https://www.taylormadegolf.com/on/demandware.static/-/Sites-tmag-master-catalog/en_US/v1784520671511/zoom/CA/M10469_zoom_D.jpg
https://www.taylormadegolf.com/on/demandware.static/-/Sites-tmag-master-catalog/en_US/v1784520671511/zoom/TC635_zoom_D.jpg
https://www.taylormadegolf.com/on/demandware.static/-/Sites-tmag-master-catalog/en_US/v1784520671511/zoom/TE674_zoom_D.jpg
https://www.taylormadegolf.com/on/demandware.static/-/Sites-tmag-master-catalog/en_US/v1784520671511/zoom/V98600_zoom_D3.jpg
https://www.taylormadegolf.com/on/demandware.static/-/Sites-tmag-master-catalog/en_US/v1784520671511/zoom/TC458_zoom_D.jpg
https://www.taylormadegolf.com/on/demandware.static/-/Sites-tmag-master-catalog/en_US/v1784520671511/zoom/M23632_zoom_D.jpg
https://www.taylormadegolf.com/on/demandware.static/-/Sites-tmag-master-catalog/en_US/v1784520671511/zoom/M31050_zoom_D.jpg
https://www.taylormadegolf.com/on/demandware.static/-/Sites-tmag-master-catalog/en_US/v1784520671511/zoom/N38558_zoom_D3.jpg
https://www.taylormadegolf.com/on/demandware.static/-/Sites-tmag-master-catalog/en_US/v1784520671511/zoom/TC563_zoom_D.jpg
https://www.taylormadegolf.com/on/demandware.static/-/Sites-tmag-master-catalog/en_US/v1784520671511/zoom/N29140_zoom_D.jpg
https://www.taylormadegolf.com/dw/image/v2/AAIS_PRD/on/demandware.static/-/Sites-tmag-master-catalog/en_US/v1784520671511/zoom/TC67W_zoom_D.jpg?sw=900&sh=900&sm=fit
https://www.taylormadegolf.com/dw/image/v2/AAIS_PRD/on/demandware.static/-/Sites-tmag-master-catalog/en_US/v1784520671511/zoom/TC67W_zoom_D2.jpg?sw=900&sh=900&sm=fit
https://www.taylormadegolf.com/dw/image/v2/AAIS_PRD/on/demandware.static/-/Sites-tmag-master-catalog/en_US/v1784520671511/zoom/TC67W_zoom_D3.jpg?sw=900&sh=900&sm=fit
https://www.taylormadegolf.com/dw/image/v2/AAIS_PRD/on/demandware.static/-/Sites-tmag-master-catalog/en_US/v1784520671511/zoom/TC67W_zoom_D4.jpg?sw=900&sh=900&sm=fit
https://www.taylormadegolf.com/on/demandware.static/-/Sites-tmag-master-catalog/en_US/v1784520671511/zoom/TC441_zoom_D.jpg
https://www.taylormadegolf.com/on/demandware.static/-/Sites-tmag-master-catalog/en_US/v1784520671511/zoom/TC448_zoom_D.jpg
https://www.taylormadegolf.com/on/demandware.static/-/Sites-tmag-master-catalog/en_US/v1784520671511/zoom/CA/M10469_zoom_D.jpg
https://www.taylormadegolf.com/on/demandware.static/-/Sites-tmag-master-catalog/en_US/v1784520671511/zoom/TC635_zoom_D.jpg
https://www.taylormadegolf.com/on/demandware.static/-/Sites-tmag-master-catalog/en_US/v1784520671511/zoom/TE674_zoom_D.jpg
https://www.taylormadegolf.com/on/demandware.static/-/Sites-tmag-master-catalog/en_US/v1784520671511/zoom/V98600_zoom_D3.jpg
https://www.taylormadegolf.com/on/demandware.static/-/Sites-tmag-master-catalog/en_US/v1784520671511/zoom/TC458_zoom_D.jpg
https://www.taylormadegolf.com/on/demandware.static/-/Sites-tmag-master-catalog/en_US/v1784520671511/zoom/M23632_zoom_D.jpg
https://www.taylormadegolf.com/on/demandware.static/-/Sites-tmag-master-catalog/en_US/v1784520671511/zoom/M31050_zoom_D.jpg
https://www.taylormadegolf.com/on/demandware.static/-/Sites-tmag-master-catalog/en_US/v1784520671511/zoom/N38558_zoom_D3.jpg
https://www.taylormadegolf.com/on/demandware.static/-/Sites-tmag-master-catalog/en_US/v1784520671511/zoom/TC563_zoom_D.jpg
https://www.taylormadegolf.com/on/demandware.static/-/Sites-tmag-master-catalog/en_US/v1784520671511/zoom/N29140_zoom_D.jpg
https://www.taylormadegolf.com/dw/image/v2/AAIS_PRD/on/demandware.static/-/Sites-tmag-master-catalog/en_US/v1784520671511/zoom/TA557W_zoom_D.jpg?sw=900&sh=900&sm=fit
https://www.taylormadegolf.com/dw/image/v2/AAIS_PRD/on/demandware.static/-/Sites-tmag-master-catalog/en_US/v1784520671511/zoom/TA557W_zoom_D2.jpg?sw=900&sh=900&sm=fit
https://www.taylormadegolf.com/dw/image/v2/AAIS_PRD/on/demandware.static/-/Sites-tmag-master-catalog/en_US/v1784520671511/zoom/TA557W_zoom_D3.jpg?sw=900&sh=900&sm=fit
https://www.taylormadegolf.com/dw/image/v2/AAIS_PRD/on/demandware.static/-/Sites-tmag-master-catalog/en_US/v1784520671511/zoom/TA557W_zoom_D5.jpg?sw=900&sh=900&sm=fit


"""

IMAGE_URLS = sorted(
    {
        url
        for line in IMAGE_URLS_TEXT.splitlines()
        if (url := line.strip()) and IMAGE_URL_PATTERN.search(url)
    }
)
if not IMAGE_URLS:
    raise RuntimeError(
        "No gallery image URLs were found. Paste the DevTools collector output into IMAGE_URLS_TEXT, then run this cell again."
    )

print(f"Found {len(IMAGE_URLS)} gallery image URLs.")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

session = requests.Session()
session.headers.update(
    {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 Chrome/142.0 Safari/537.36"
        ),
        "Referer": "https://www.taylormadegolf.com/",
    }
)

for index, image_url in enumerate(IMAGE_URLS, start=1):
    filename = Path(urlparse(image_url).path).name or f"image_{index}.jpg"
    destination = OUTPUT_DIR / filename
    if destination.exists():
        destination = OUTPUT_DIR / f"{Path(filename).stem}_{index}{Path(filename).suffix or '.jpg'}"

    try:
        response = session.get(image_url, timeout=30)
        response.raise_for_status()
        destination.write_bytes(response.content)
        print(f"Downloaded: {destination}")
    except requests.RequestException as error:
        print(f"Failed: {image_url}\nReason: {error}")


Found 32 gallery image URLs.
Downloaded: C:\Users\Matt Shaver\OneDrive\Desktop\SwingSight-AI\assets\club-reference-images\iron\wedges\N29140_zoom_D.jpg
Downloaded: C:\Users\Matt Shaver\OneDrive\Desktop\SwingSight-AI\assets\club-reference-images\iron\wedges\N29140_zoom_D2.jpg
Downloaded: C:\Users\Matt Shaver\OneDrive\Desktop\SwingSight-AI\assets\club-reference-images\iron\wedges\N29140_zoom_D3.jpg
Downloaded: C:\Users\Matt Shaver\OneDrive\Desktop\SwingSight-AI\assets\club-reference-images\iron\wedges\N29140_zoom_D4.jpg
Downloaded: C:\Users\Matt Shaver\OneDrive\Desktop\SwingSight-AI\assets\club-reference-images\iron\wedges\TA557W_zoom_D.jpg
Downloaded: C:\Users\Matt Shaver\OneDrive\Desktop\SwingSight-AI\assets\club-reference-images\iron\wedges\TA557W_zoom_D2.jpg
Downloaded: C:\Users\Matt Shaver\OneDrive\Desktop\SwingSight-AI\assets\club-reference-images\iron\wedges\TA557W_zoom_D3.jpg
Downloaded: C:\Users\Matt Shaver\OneDrive\Desktop\SwingSight-AI\assets\club-reference-images\iron\wedges\